# Real-data validation, Part 2: extension to real, complex SOA via molecular proxies

Reproduces the code behind paper Section 3.5: validating the ML-Tg + AIOMFAC + VTF pipeline against real
chamber-generated SOA (not pure compounds) from three precursor systems -- alpha-pinene, isoprene, toluene --
using literature molecular proxies, plus the toluene proxy-sensitivity check (Table 2 in the paper) that shows
proxy-molecule choice, not an inherent pipeline limitation, was the main driver of an initially poor toluene
result.

**Data:** the "Aqueous SOA Systems" folder of the Gervasi, Topping and Zuend (2020) Supplement -- 9 independent
alpha-pinene SOA chamber studies, 2 isoprene SOA studies (Song et al., 2015), 2 toluene SOA studies (Song et
al., 2016).

## 0. Setup

In [1]:
import os, sys
if "google.colab" in sys.modules:   # Colab: install the package; locally: use the installed aiomfac_py
    if not os.path.exists("aiomfac-python"):
        !git clone -q https://github.com/SeoulTechPSE/aiomfac-python.git
    %cd aiomfac-python
    !python setup_aiomfac.py --smiles --no-editable -q 2>&1 | tail -20
    !pip install -q -e ".[tgml,tgml-smiles]" 2>&1 | tail -10
    !pip install -q scipy matplotlib
    %cd ..
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
import aiomfac_py
print("setup done; aiomfac_py", aiomfac_py.__version__, "from", os.path.dirname(aiomfac_py.__file__))

setup done; aiomfac_py 1.3.0 from /Users/kyyoo/Downloads/aiomfac_/aiomfac-python/src/aiomfac_py


In [2]:
# Upload gervasi_supp.zip (Colab only) -- contains the "Aqueous SOA Systems" folder used below.
try:
    from google.colab import files
    if not os.path.exists("Aqueous SOA Systems"):
        print("Select gervasi_supp.zip to upload...")
        uploaded = files.upload()
        import zipfile
        with zipfile.ZipFile("gervasi_supp.zip") as zf:
            zf.extractall(".")
except ImportError:
    print("Not running in Colab -- make sure the 'Aqueous SOA Systems' folder is in the current directory "
          "(unzip gervasi_supp.zip here).")
print(os.listdir("."))

Not running in Colab -- make sure the 'Aqueous SOA Systems' folder is in the current directory (unzip gervasi_supp.zip here).
['glassy_regime_diagnosis.json', 'fig_C_tg_validation.png', 'gervasi_supp.zip', 'aiomfac_validation_01_pure_compounds.ipynb', 'aiomfac_validation_02_soa_proxies.ipynb', 'bimog_unified.json', 'log_01_pure_compounds.txt', 'fig_B_example_curves.png', 'validation_results.json', 'literature_curves.json', 'Aqueous SOA Systems', 'Aqueous Binary Systems', 'fig_A_predicted_vs_measured.png', 'log_02_soa_proxies.txt']


In [3]:
import sys
sys.path.insert(0, "aiomfac-python/src")
import glob, os, re
import numpy as np
from aiomfac_py.s2as import smiles_to_components
from aiomfac_py import ActivityModel, Component
from aiomfac_py.tgml_armeli import predict_tg_ml_smiles
from aiomfac_py.viscosity import organic_mixture_viscosity, pure_organic_viscosity_vtf

WATER = Component(1, "Water", ((16, 1),))
SOA_DIR = "Aqueous SOA Systems"


def parse_soa_file(path):
    with open(path, encoding="utf-8-sig") as f:
        lines = [l.strip().strip('"') for l in f.readlines()]
    m = re.search(r"T\s*=\s*([\d.]+)\s*K", lines[0])
    T_K = float(m.group(1)) if m else None
    pts = []
    for line in lines[4:]:
        parts = [p for p in re.split(r"[,\t]", line) if p.strip() != ""]
        if len(parts) < 2:
            continue
        try:
            rh, eta = float(parts[0]), float(parts[1])
        except ValueError:
            continue
        if eta > 0 and 0 < rh < 100:
            pts.append((rh / 100.0, np.log10(eta)))
    return T_K, pts


def find_wtf_water_for_aw(model, target_aw, T_K, lo=0.0005, hi=0.9995, tol=1e-3, max_iter=60):
    def aw_at(wtf_w):
        res = model.evaluate([wtf_w, 1.0 - wtf_w], T_K, basis="mass")
        return float(res.activity[0])
    aw_lo, aw_hi = aw_at(lo), aw_at(hi)
    if not (aw_lo <= target_aw <= aw_hi):
        return None
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        if abs(aw_at(mid) - target_aw) < tol:
            return mid
        if aw_at(mid) < target_aw:
            lo = mid
        else:
            hi = mid
    return mid


def predict_eta_curve(smiles, tg_K, T_K, aw_targets):
    r = smiles_to_components([smiles], water_as_component1=True)
    organic = r.components[1]
    model = ActivityModel([WATER, Component(2, "organic", organic.subgroups)])
    eta0 = pure_organic_viscosity_vtf(T_K, tg_K)
    preds = []
    for aw in aw_targets:
        wtf_w = find_wtf_water_for_aw(model, aw, T_K)
        if wtf_w is None:
            preds.append(np.nan); continue
        res = model.evaluate([wtf_w, 1.0 - wtf_w], T_K, basis="mass")
        v = organic_mixture_viscosity(model, res.x, T_K, {2: eta0})
        eta = v.eta_pas
        preds.append(np.log10(eta) if (np.isfinite(eta) and eta > 0) else np.nan)
    return np.array(preds)

print("shared helper functions defined")

shared helper functions defined


## 1. Alpha-pinene SOA: pinonic-acid / pinic-acid proxy validation (paper Section 3.5, Table 1 row 1)

Compares the field's standard early- vs. aged-generation proxy pair for alpha-pinene SOA against 9 independent
real chamber studies.

In [4]:
PROXIES_APINENE = {
    "pinonic_acid": {"smiles": "CC(=O)C1CC(C1(C)C)CC(=O)O", "o_to_c": 0.30, "tg_measured_K": 265.0},
    "pinic_acid": {"smiles": "CC1(C(CC1C(=O)O)CC(=O)O)C", "o_to_c": 0.4444, "tg_measured_K": 268.1},
}
for name, info in PROXIES_APINENE.items():
    tg_ml = predict_tg_ml_smiles(info["smiles"]).tg_K
    print(f"{name}: O:C={info['o_to_c']}  measured Tg={info['tg_measured_K']:.1f} K  ML-predicted Tg={tg_ml:.1f} K")

print("\n=== real chamber alpha-pinene SOA vs. pinonic-acid / pinic-acid proxy predictions ===")
files_ap = sorted(glob.glob(os.path.join(SOA_DIR, "*aPinene_SOA*.txt")))
apinene_rows = []
for path in files_ap:
    fname = os.path.basename(path)
    study = fname.split("aPinene_SOA_")[1].rsplit(".txt", 1)[0]
    T_K, pts = parse_soa_file(path)
    if not pts or T_K is None:
        continue
    aw_vals = np.array([p[0] for p in pts])
    y_meas = np.array([p[1] for p in pts])
    preds = {}
    for name, info in PROXIES_APINENE.items():
        tg_ml = predict_tg_ml_smiles(info["smiles"]).tg_K
        preds[name + "_ml"] = predict_eta_curve(info["smiles"], tg_ml, T_K, aw_vals)
    print(f"\n  {study:30s} T={T_K:.1f}K  n={len(pts)}  measured mean={y_meas.mean():+.2f}")
    for key, pred in preds.items():
        valid = np.isfinite(pred)
        mae = float(np.mean(np.abs(pred[valid] - y_meas[valid]))) if valid.any() else None
        print(f"    {key:16s} predicted mean={np.nanmean(pred):+.2f}  MAE vs real SOA={mae:.2f}")
        apinene_rows.append({"study": study, "proxy": key, "mae": mae})

maes_pinonic = [r["mae"] for r in apinene_rows if r["proxy"] == "pinonic_acid_ml"]
maes_pinic = [r["mae"] for r in apinene_rows if r["proxy"] == "pinic_acid_ml"]
print(f"\npinonic_acid mean MAE = {np.mean(maes_pinonic):.3f}   pinic_acid mean MAE = {np.mean(maes_pinic):.3f}")

pinonic_acid: O:C=0.3  measured Tg=265.0 K  ML-predicted Tg=267.8 K
pinic_acid: O:C=0.4444  measured Tg=268.1 K  ML-predicted Tg=271.0 K

=== real chamber alpha-pinene SOA vs. pinonic-acid / pinic-acid proxy predictions ===

  293K_Abramson2013              T=293.1K  n=1  measured mean=+7.05
    pinonic_acid_ml  predicted mean=+6.41  MAE vs real SOA=0.64
    pinic_acid_ml    predicted mean=+6.80  MAE vs real SOA=0.25

  293K_Pajunoja2014              T=293.1K  n=1  measured mean=+11.02
    pinonic_acid_ml  predicted mean=+5.91  MAE vs real SOA=5.11
    pinic_acid_ml    predicted mean=+6.01  MAE vs real SOA=5.01



  293K_Zhang2015                 T=293.1K  n=36  measured mean=+7.66
    pinonic_acid_ml  predicted mean=+5.12  MAE vs real SOA=2.54
    pinic_acid_ml    predicted mean=+4.89  MAE vs real SOA=2.78

  294K_Grayson2016@121ugm-3      T=294.0K  n=4  measured mean=+4.67
    pinonic_acid_ml  predicted mean=+4.59  MAE vs real SOA=0.64
    pinic_acid_ml    predicted mean=+4.18  MAE vs real SOA=1.08

  294K_Grayson2016@520ugm-3      T=294.0K  n=4  measured mean=+5.07
    pinonic_acid_ml  predicted mean=+4.59  MAE vs real SOA=0.48
    pinic_acid_ml    predicted mean=+4.18  MAE vs real SOA=0.89

  294K_Renbaum-Wolff2013@liquid  T=294.0K  n=8  measured mean=+1.91
    pinonic_acid_ml  predicted mean=+1.59  MAE vs real SOA=0.34
    pinic_acid_ml    predicted mean=+0.31  MAE vs real SOA=1.60



  294K_Renbaum-Wolff2013@semisolid T=294.0K  n=9  measured mean=+5.39
    pinonic_acid_ml  predicted mean=+3.67  MAE vs real SOA=1.71
    pinic_acid_ml    predicted mean=+2.89  MAE vs real SOA=2.50

  297K_Kidd2014                  T=297.0K  n=1  measured mean=+3.67
    pinonic_acid_ml  predicted mean=+0.89  MAE vs real SOA=2.78
    pinic_acid_ml    predicted mean=-0.41  MAE vs real SOA=4.09

  298K_Bateman2015               T=298.1K  n=1  measured mean=+2.03
    pinonic_acid_ml  predicted mean=+1.86  MAE vs real SOA=0.17
    pinic_acid_ml    predicted mean=+0.72  MAE vs real SOA=1.31

pinonic_acid mean MAE = 1.602   pinic_acid mean MAE = 2.167


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 11, "font.family": "sans-serif", "axes.spines.top": False,
                      "axes.spines.right": False, "figure.dpi": 150})

fig, ax = plt.subplots(figsize=(6.5, 5.2))
cmap = matplotlib.colormaps["tab10"].resampled(len(files_ap))
for i, path in enumerate(files_ap):
    study = os.path.basename(path).split("aPinene_SOA_")[1].rsplit(".txt", 1)[0].replace("_", " ")
    T_K, pts = parse_soa_file(path)
    if not pts or T_K is None:
        continue
    rh = np.array([p[0] for p in pts]) * 100.0
    eta = np.array([p[1] for p in pts])
    ax.scatter(rh, eta, s=26, color=cmap(i), label=study, alpha=0.85, zorder=5)

rh_grid = np.linspace(1, 95, 60)
aw_grid = rh_grid / 100.0
T_ref = 294.0
colors = {"pinonic_acid": "#2b6cb0", "pinic_acid": "#38a169"}
for name, info in PROXIES_APINENE.items():
    pred = predict_eta_curve(info["smiles"], info["tg_measured_K"], T_ref, aw_grid)
    ax.plot(rh_grid, pred, "-", lw=2.5, color=colors[name], label=f"{name} (measured Tg)", zorder=3)

ax.set_xlabel("relative humidity [%]")
ax.set_ylabel(r"$\log_{10}(\eta$ / Pa s$)$")
ax.set_title(r"$\alpha$-pinene SOA: real chamber measurements vs."
             "\nsingle-proxy-molecule pipeline predictions (294 K)", fontsize=10.5)
ax.legend(fontsize=6.8, frameon=False, loc="upper right", ncol=1)
fig.tight_layout()
fig.savefig("fig_D_apinene_soa.png", dpi=220)
plt.show()
print("wrote fig_D_apinene_soa.png")

wrote fig_D_apinene_soa.png


/var/folders/4x/8kn2nym12cn7x7qmg_6s4b8h0000gn/T/ipykernel_55172/2942097609.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Isoprene SOA and initial (ad hoc) toluene SOA proxies (paper Section 3.5)

Isoprene SOA: 2-methyltetrol, the standard low-NOx GC-MS tracer. Toluene SOA: an initial pass with two
chemically plausible but ad hoc candidates (4-methylcatechol, glutaric acid) -- which Section 3 below shows was
NOT the best choice.

In [6]:
SYSTEMS = {
    "isoprene": {"glob": "*isoprene_SOA*.txt", "proxies": {"2-methyltetrol": {"smiles": "OCC(C)(O)C(O)CO"}}},
    "toluene": {"glob": "*toluene_SOA*.txt", "proxies": {"4-methylcatechol": {"smiles": "Cc1ccc(O)c(O)c1"},
                                                          "glutaric_acid": {"smiles": "OC(=O)CCCC(=O)O"}}},
}

all_rows = []
for sysname, sysinfo in SYSTEMS.items():
    print(f"\n{'='*60}\n{sysname.upper()} SOA\n{'='*60}")
    for pname, pinfo in sysinfo["proxies"].items():
        tg_ml = predict_tg_ml_smiles(pinfo["smiles"]).tg_K
        pinfo["tg_ml_K"] = tg_ml
        print(f"  proxy {pname}: ML-predicted Tg = {tg_ml:.1f} K")

    files_s = sorted(glob.glob(os.path.join(SOA_DIR, sysinfo["glob"])))
    for path in files_s:
        fname = os.path.basename(path)
        study = fname.split(f"{sysname}_SOA_")[1].rsplit(".txt", 1)[0]
        T_K, pts = parse_soa_file(path)
        if not pts or T_K is None:
            continue
        aw_vals = np.array([p[0] for p in pts])
        y_meas = np.array([p[1] for p in pts])
        print(f"\n  {study:30s} T={T_K:.1f}K  n={len(pts)}  measured mean={y_meas.mean():+.2f}")
        for pname, pinfo in sysinfo["proxies"].items():
            pred, _ = predict_eta_curve(pinfo["smiles"], pinfo["tg_ml_K"], T_K, aw_vals), None
            valid = np.isfinite(pred)
            mae = float(np.mean(np.abs(pred[valid] - y_meas[valid]))) if valid.any() else None
            print(f"    {pname:20s} predicted mean={np.nanmean(pred):+.2f}  MAE={mae:.2f}")
            all_rows.append({"system": sysname, "study": study, "proxy": pname, "mae": mae})

print("\n=== summary ===")
for sysname in SYSTEMS:
    by_proxy = {}
    for r in all_rows:
        if r["system"] == sysname and r["mae"] is not None:
            by_proxy.setdefault(r["proxy"], []).append(r["mae"])
    for pname, maes in by_proxy.items():
        print(f"{sysname} / {pname}: mean MAE = {np.mean(maes):.2f} (n={len(maes)} curves)")


ISOPRENE SOA
  proxy 2-methyltetrol: ML-predicted Tg = 246.1 K

  295K_Song2015@liquid           T=295.0K  n=5  measured mean=+0.35
    2-methyltetrol       predicted mean=-1.40  MAE=1.76

  295K_Song2015@semisolid        T=295.0K  n=4  measured mean=+3.96
    2-methyltetrol       predicted mean=+2.28  MAE=1.68

TOLUENE SOA
  proxy 4-methylcatechol: ML-predicted Tg = 242.0 K
  proxy glutaric_acid: ML-predicted Tg = 252.5 K

  295K_Song2016@liquidsemisolid  T=295.0K  n=25  measured mean=+2.63
    4-methylcatechol     predicted mean=-1.01  MAE=3.64
    glutaric_acid        predicted mean=-0.24  MAE=2.86



  295K_Song2016@solid            T=294.0K  n=4  measured mean=+8.58


    4-methylcatechol     predicted mean=+1.63  MAE=6.95
    glutaric_acid        predicted mean=+2.80  MAE=5.78

=== summary ===
isoprene / 2-methyltetrol: mean MAE = 1.72 (n=2 curves)
toluene / 4-methylcatechol: mean MAE = 5.29 (n=2 curves)
toluene / glutaric_acid: mean MAE = 4.32 (n=2 curves)


## 3. Toluene-SOA proxy-molecule sensitivity check (paper Table 2)

Tests 5 candidate proxies spanning low-to-high fragility/Tg, including DHOPA (2,3-dihydroxy-4-oxopentanoic
acid, the field-established GC-MS tracer for toluene SOA, nominated *a priori*) and 4-methyl-5-nitrocatechol (a
literature-documented 2nd-generation tracer, found to perform best but identified *post hoc* -- see the paper's
discussion of why DHOPA, not this one, is reported as the primary result).

In [7]:
PROXIES_TOLUENE = {
    "4-methylcatechol": "Cc1ccc(O)c(O)c1",
    "glutaric_acid": "OC(=O)CCCC(=O)O",
    "DHOPA": "CC(=O)C(O)C(O)C(=O)O",
    "2-methylglutaric_acid": "OC(=O)C(C)CCC(=O)O",
    "4-methyl-5-nitrocatechol": "Cc1cc(O)c(O)cc1[N+](=O)[O-]",
}

print("=== candidate proxy Tg (ML) ===")
tg_ml_tol = {}
for name, smi in PROXIES_TOLUENE.items():
    r = smiles_to_components([smi], water_as_component1=True)
    if r.removed:
        print(f"  {name:28s} NOT DECOMPOSABLE"); continue
    tg = predict_tg_ml_smiles(smi).tg_K
    tg_ml_tol[name] = tg
    print(f"  {name:28s} ML Tg = {tg:6.1f} K")

files_tol = sorted(glob.glob(os.path.join(SOA_DIR, "*toluene_SOA*.txt")))
print("\n=== per-curve MAE, all candidate proxies ===")
for path in files_tol:
    fname = os.path.basename(path)
    study = fname.split("toluene_SOA_")[1].rsplit(".txt", 1)[0]
    T_K, pts = parse_soa_file(path)
    aw_vals = np.array([p[0] for p in pts])
    y_meas = np.array([p[1] for p in pts])
    print(f"\n  {study:30s} T={T_K:.1f}K n={len(pts)} measured mean={y_meas.mean():+.2f}")
    for name, tg in tg_ml_tol.items():
        pred = predict_eta_curve(PROXIES_TOLUENE[name], tg, T_K, aw_vals)
        valid = np.isfinite(pred)
        mae = float(np.mean(np.abs(pred[valid] - y_meas[valid]))) if valid.any() else None
        print(f"    {name:28s} Tg={tg:6.1f}K  pred mean={np.nanmean(pred):+.2f}  MAE={mae:.2f}" if mae is not None
              else f"    {name:28s} all-NaN")

=== candidate proxy Tg (ML) ===
  4-methylcatechol             ML Tg =  242.0 K
  glutaric_acid                ML Tg =  252.5 K
  DHOPA                        ML Tg =  285.3 K
  2-methylglutaric_acid        ML Tg =  262.2 K
  4-methyl-5-nitrocatechol     ML Tg =  299.6 K

=== per-curve MAE, all candidate proxies ===

  295K_Song2016@liquidsemisolid  T=295.0K n=25 measured mean=+2.63
    4-methylcatechol             Tg= 242.0K  pred mean=-1.01  MAE=3.64


    glutaric_acid                Tg= 252.5K  pred mean=-0.24  MAE=2.86


    DHOPA                        Tg= 285.3K  pred mean=+0.75  MAE=1.88


    2-methylglutaric_acid        Tg= 262.2K  pred mean=+0.49  MAE=2.14
    4-methyl-5-nitrocatechol     Tg= 299.6K  pred mean=+3.28  MAE=0.66



  295K_Song2016@solid            T=294.0K n=4 measured mean=+8.58
    4-methylcatechol             Tg= 242.0K  pred mean=+1.63  MAE=6.95
    glutaric_acid                Tg= 252.5K  pred mean=+2.80  MAE=5.78
    DHOPA                        Tg= 285.3K  pred mean=+6.31  MAE=2.27
    2-methylglutaric_acid        Tg= 262.2K  pred mean=+3.96  MAE=4.61
    4-methyl-5-nitrocatechol     Tg= 299.6K  pred mean=+9.69  MAE=1.11


## 4. Download results

In [8]:
import json
with open("soa_validation_results.json", "w") as f:
    json.dump({"apinene": apinene_rows, "isoprene_and_toluene_initial": all_rows}, f, indent=1)

outputs = ["soa_validation_results.json", "fig_D_apinene_soa.png"]
try:
    from google.colab import files
    for fn in outputs:
        if os.path.exists(fn):
            files.download(fn)
except ImportError:
    print("Not running in Colab -- output files are in the current directory:", outputs)

Not running in Colab -- output files are in the current directory: ['soa_validation_results.json', 'fig_D_apinene_soa.png']
